# 실습 1-1 · 파이썬에서 LLM 호출하기 (STEP 1~7)

LLM 호출 절차
1. 접속 정보: 어디로(`base_url`) · 누구로(`api_key`) · 어떤 모델(`model`)
2. 클라이언트: `OpenAI(...)` 객체 = 서버와 통신하는 창구
3. 메시지: `[{"role": 역할, "content": 내용}]`
4. 요청: `client.chat.completions.create(...)`
5. 답 꺼내기: `response.choices[0].message.content`

Ollama와 MonoRouter는 ① 접속 정보만 다르고 ②~⑤는 같습니다. 셀을 위에서부터 Shift+Enter로 하나씩 실행합니다.

## STEP 1. 접속 정보와 클라이언트 (①②)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "llm_client.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing llm_client.py")
sys.path.insert(0, str(PROJECT_ROOT))

from llm_client import create_llm_client

PROVIDER = "mono"
client, MODEL = create_llm_client(provider=PROVIDER, api_key_env="MY_API_KEY")
print(PROVIDER, MODEL)

## STEP 2. 가장 짧은 호출: user 메시지 하나 (③④⑤)
오류가 나면 Ollama 실행 여부(`ollama list`)와 모델 이름, 또는 API 키를 확인합니다.

In [2]:
response = client.chat.completions.create(          # ④ 요청 (답이 올 때까지 기다림)
    model=MODEL,
    messages=[{"role": "user", "content": "SOC가 무엇인지 한 문장으로 설명해 줘."}],   # ③
)
print(response.choices[0].message.content)          # ⑤ 답 꺼내기

SOC는 문맥에 따라 사회간접자본, 시스템온칩(System on Chip), 보안관제센터(Security Operations Center) 등을 뜻하는 약어입니다.


## STEP 3. 응답 객체에는 답 말고도 정보가 들어 있다

In [3]:
import time

start = time.time()
response = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "피싱을 한 문장으로 설명해 줘."}])
choice = response.choices[0]

print("답        :", choice.message.content)
print("응답 모델  :", response.model)
print("끝난 이유  :", choice.finish_reason)      # stop=정상 종료, length=길이 제한으로 잘림
print("토큰      :", response.usage)              # 입력 · 출력 토큰 수 (비용 · 속도의 기준)
print(f"걸린 시간  : {time.time() - start:.1f}초")

답        : 피싱은 공격자가 신뢰할 수 있는 기관이나 사람을 사칭해 비밀번호, 금융정보 등 민감한 정보를 속여 빼내는 사이버 사기입니다.
응답 모델  : gpt-5.5-2026-04-23
끝난 이유  : stop
토큰      : CompletionUsage(completion_tokens=58, prompt_tokens=18, total_tokens=76, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=10, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))
걸린 시간  : 2.8초


## STEP 4. 같은 질문, 역할(system) 없음 vs 있음

In [5]:
QUESTION = "로그인 실패가 1분에 50번 났어. 어떻게 봐야 해?"

plain = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": QUESTION}])

with_role = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": "당신은 금융기관 보안관제(SOC) 분석가입니다. 세 줄 이내로 답합니다."},
              {"role": "user", "content": QUESTION}])

print("[역할 없음]\n" + plain.choices[0].message.content)
print("\n[역할 있음]\n" + with_role.choices[0].message.content)

[역할 없음]
1분에 로그인 실패 50번이면 **단순 사용자 실수보다는 공격/장애 가능성을 우선 의심**해야 합니다. 다만 “누가, 어디서, 어떤 계정에” 실패했는지에 따라 해석이 달라집니다.

## 1. 먼저 이렇게 분류해서 보세요

### A. 하나의 IP에서 여러 계정으로 실패
- 예: `1.2.3.4`에서 50개 계정에 로그인 실패
- 가능성: **Password spraying / 계정 목록 대입 공격**
- 특히 같은 비밀번호를 여러 계정에 시도하는 패턴일 수 있습니다.

### B. 하나의 계정에 여러 IP에서 실패
- 예: `admin@example.com`에 여러 국가/IP에서 실패
- 가능성: **Credential stuffing / 특정 계정 탈취 시도**
- 유출된 비밀번호를 봇넷으로 시도하는 경우가 많습니다.

### C. 하나의 IP에서 하나의 계정에 반복 실패
- 예: 한 사용자가 1분에 50번 실패
- 가능성:
  - 사용자의 저장된 비밀번호 오류
  - 스크립트/앱 연동 오류
  - 무차별 대입 공격
- 내부 IP나 사내 VPN이면 장애/설정 문제도 봐야 합니다.

### D. 실패 후 성공이 있음
- 가장 중요합니다.
- 예: 49번 실패 후 1번 성공
- 가능성: **계정 탈취 성공**
- 이 경우 해당 계정은 즉시 조사/보호 대상입니다.

---

## 2. 확인해야 할 핵심 항목

로그에서 아래를 확인하세요.

- 실패한 계정 수
- 출발지 IP 수
- User-Agent
- 국가/지역
- 실패 사유
  - 비밀번호 오류
  - 존재하지 않는 계정
  - MFA 실패
  - 계정 잠김
- 같은 시간대 성공 로그인 여부
- 정상 대비 증가율
- 관리자/권한 계정 포함 여부
- VPN, 프록시, 클라우드 IP 여부

---

## 3. 판단 기준

### 공격 가능성이 높은 경우
- 짧은 시간에 다수 계정 대상
- 여러 국가/IP에서 동시 발생
- 존재하지 않는 계정에도 시도
- User-Agent가 비정상적이거나 동일함
-

## STEP 5. temperature: 답이 흔들리는 정도
같은 질문을 두 번씩 보냅니다. 탐지 · 분류처럼 일관성이 중요한 일은 0~0.2로 낮게 씁니다.

In [6]:
q = "보안 관제 요원이 사내에서 겪을 수 있는 보안 사고 시나리오 3가지만 자유롭게 지어내 줘. 각 시나리오는 1줄씩"
for temp in (0, 1.0):
    names = []
    for _ in range(2):
        r = client.chat.completions.create(model=MODEL, temperature=temp,
                                           messages=[{"role": "user", "content": q}])
        names.append(r.choices[0].message.content.strip()[:250])
    print(f"temperature={temp}: {names}")

temperature=0: ['1. 야간 시간대에 특정 서버에서 비정상적인 대량 파일 암호화 행위가 탐지되어 랜섬웨어 감염이 의심되는 상황.  \n2. 임직원이 사칭 메일의 링크를 클릭한 뒤 계정이 탈취되어 사내 VPN에서 해외 IP 로그인이 발생한 상황.  \n3. 퇴사 예정 직원 계정으로 대량의 내부 문서가 클라우드 저장소에 업로드되어 정보 유출이 의심되는 상황.', '1. 새벽 시간대에 임직원 계정으로 해외 IP에서 VPN 접속 후 대량의 내부 문서 다운로드가 탐지된다.  \n2. 사내 PC 여러 대에서 동일한 랜섬웨어 의심 프로세스가 실행되며 파일 확장자가 일괄 변경되는 이벤트가 발생한다.  \n3. 개발 서버에서 평소 없던 관리자 계정이 생성되고 외부 저장소로 소스코드 업로드 트래픽이 확인된다.']
temperature=1.0: ['1. 새벽 시간대에 임직원 계정으로 해외 IP에서 비정상 로그인 시도가 급증해 계정 탈취 여부를 조사하는 상황.  \n2. 내부 PC 한 대에서 랜섬웨어 의심 행위가 탐지되어 네트워크 격리 및 감염 범위 확인을 진행하는 상황.  \n3. 퇴사 예정 직원이 대량의 사내 문서를 외부 클라우드로 업로드하려 해 데이터 유출 여부를 분석하는 상황.', '1. 퇴근 시간 이후 내부 계정으로 대량의 고객 정보 조회가 발생해 계정 탈취 또는 내부자 유출이 의심되는 상황.  \n2. 사내 직원이 피싱 메일의 첨부파일을 실행한 뒤 PC에서 랜섬웨어 행위와 파일 암호화 시도가 탐지되는 상황.  \n3. 외부에서 VPN으로 접속한 계정이 평소와 다른 국가·시간대에서 로그인 후 중요 서버에 반복 접근하는 상황.']


## STEP 6. 대화 이어가기
LLM은 이전 대화를 기억하지 않습니다. 앞의 질문과 답을 `messages`에 다시 넣어 보냅니다.

In [ ]:
messages = [{"role": "system", "content": "당신은 금융기관 SOC 분석가입니다. 짧게 2줄 이내로 답합니다."},
            {"role": "user", "content": "DoS 공격이 뭐야?"}]
a1 = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content    #질문하기
print("A1:", a1)                                                                  # 답변 출력

messages.append({"role": "assistant", "content": a1})                             # assistant 모델의 답을 추가  
messages.append({"role": "user", "content": "그럼 그걸 로그에서 어떻게 알아봐?"})    # '그걸' = 앞 대화에 
a2 = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content  #질문 이어서 하기 
print("\nA2:", a2)                                                                # 답변 출력

## STEP 7. 함수로 묶기: 이후 실습의 기본 틀
`ask()` 하나로 질문을 보내고 답을 받습니다. 로그는 UNSW-NB15 원본 레코드입니다.

In [ ]:
def ask(prompt, system="당신은 금융기관 SOC 분석가입니다.", temperature=0.2):
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}])
    return r.choices[0].message.content

log = "src=175.45.176.1:20549 dst=149.171.126.10:111 proto=udp service=- state=INT dur=0.000 spkts=2 dpkts=0"
print(log)
print(ask(f"이 로그 위험해? 이유를 두 줄로 써 줘.\n{log}"))

다음 실습 2a에서는 이 프롬프트를 역할 → 맥락 → 제약 → 출력 형식 순서로 구체화합니다.